In [1]:
# LangChain components to use
from langchain_community.vectorstores import Cassandra
from langchain_openai import OpenAI
from langchain_openai import OpenAIEmbeddings
# Support for dataset retrieval with Hugging Face
from datasets import load_dataset
# With CassIO, the engine powering the Astra DB integration in LangChain,
# you will also initialize the DB connection:
import cassio

e:\00-My_github_repository\GenAiApps\LangChain_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
import os
from dotenv import load_dotenv
load_dotenv() # load all environment variables
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["ASTRA_DB_TOCKEN"] = os.getenv("PDF_QUERY_ASTRA_DB_TOCKEN")
os.environ["ASTRA_DB_ID"] = os.getenv("PDF_QUERY_ASTRA_DB_ID")
# LangSmith Tracking configuration
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY_SIMPLE_PDF_QUERY")
os.environ["LANGCHAIN_TRACKING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT_SIMPLE_PDF_QUERY")
astra_db_id = os.getenv("PDF_QUERY_ASTRA_DB_ID")
astra_db_token = os.getenv("PDF_QUERY_ASTRA_DB_TOCKEN")
openai_api_key = os.getenv("OPENAI_API_KEY")

In [24]:
## Helper functions
## Function to format retrieved documents for the LLM
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [5]:
## Load document
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("AUTOSAR_SWS_CANInterface.pdf")
documents = loader.load()
## Split document into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 150,
    separators = ["\n\n", "\n• ", "\n- ", "\n", ". ", "! ", "? ", "; ", ", ", " ", ""]
)
split_documents = splitter.split_documents(documents)


In [6]:
from PyPDF2 import PdfReader
from typing_extensions import Concatenate
pdf_reader = PdfReader("AUTOSAR_SWS_CANInterface.pdf")
## read text from document
raw_text = ''
for i,page in enumerate(pdf_reader.pages):
    text_content = page.extract_text()
    raw_text += text_content

In [8]:
## Connect to Astra DB
cassio.init(database_id=astra_db_id, token=astra_db_token)

In [10]:
## Create llm model and embedding instances
llm_model = OpenAI(openai_api_key = openai_api_key)
embedding = OpenAIEmbeddings(openai_api_key = openai_api_key)

In [20]:
vector_store = Cassandra(embedding = embedding, table_name = "qa_mini", session = None, keyspace = None)

In [19]:
## Split document into chunks
from langchain_text_splitters import CharacterTextSplitter
text_splitter = CharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 150,
    separator = "\n",
    length_function = len
)
split_text= text_splitter.split_text(raw_text)

In [21]:
## Embed the chunks and add to vector store
vector_store.add_texts(split_text)
retriever = vector_store.as_retriever()

In [25]:

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
prompt = ChatPromptTemplate.from_template(
"""Answer the question using only the context.

Context:
{context}

Question: {question}
"""
)
## Create the RAG chain
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm_model
    | StrOutputParser()
)

response = rag_chain.invoke("What is Hardware object handles?")
print(response)


Answer: Hardware object handles (HOH) for transmission (HTH) and reception (HRH) are abstract references to CAN mailbox structures that contain CAN related parameters. Each HOH represents one or multiple CAN hardware objects and is used as a parameter by the CAN Interface Layer for software filtering. The HTH and HRH are defined and provided by the CAN Driver and are used to prevent priority inversion during transmission of high-priority L-PDUs. A CAN hardware object is a PDU buffer inside the CAN RAM of the CAN Hardware Unit/Controller. 
